# 07 - Document ingestion for base RAG

This notebook shows the base Release 02 ingestion flow: load markdown documents, inspect metadata, split into chunks, build embeddings, persist a Chroma index, and close with a light 2D visualization.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.manifold import TSNE

from collections import Counter

from chromadb import PersistentClient

from llmkit.config.settings import get_settings
from llmkit.rag.loaders import load_markdown_documents
from llmkit.rag.splitters import split_documents
from llmkit.rag.vectorstores import build_chroma_index, read_index_metadata

settings = get_settings()
KNOWLEDGE_BASE = settings.knowledge_base_dir
VECTORSTORE_PATH = settings.vectorstore_dir / "local_docs"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

## 1. Load documents manually

In [ ]:
documents = load_markdown_documents(KNOWLEDGE_BASE)
print(f"Loaded documents: {len(documents)}")
print(Counter(document.metadata["doc_type"] for document in documents))
documents[0]

## 2. Split documents into chunks

In [ ]:
chunks = split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"Chunks created: {len(chunks)}")
chunks[0]

## 3. Build a persistent Chroma index

In [ ]:
build_chroma_index(
    chunks,
    persist_directory=VECTORSTORE_PATH,
    source_path=str(KNOWLEDGE_BASE),
)
read_index_metadata(VECTORSTORE_PATH)

## 4. Light 2D visualization

This is only a quick sanity check for the base release, not a full diagnostics notebook.

In [ ]:
collection = PersistentClient(path=str(VECTORSTORE_PATH)).get_or_create_collection("documents")
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents_text = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["doc_type"] for metadata in metadatas]

reduced_vectors = TSNE(n_components=2, random_state=42).fit_transform(vectors)
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=8, opacity=0.8),
            text=[f"Type: {t}<br>Text: {d[:120]}..." for t, d in zip(doc_types, documents_text)],
            hoverinfo="text",
        )
    ]
)
fig.update_layout(title="Base RAG vector visualization", width=850, height=600)
fig.show()